# ML-11 — Capstone Research Paper

**Lane:** Refresh / Content Opportunity Scoring
**Author:** [Intern Name]
**Repo:** https://github.com/[username]/flyrank-ml-internship
**Date:** 2026

## 1. Title + Abstract

**Abstract** (5 sentences: question → method → result → what it's for):

1. **Question:** Which safe content/search signals are associated with visibility decline in search results, so that FlyRank editors can prioritize content refresh actions without client-identifying data or causal claims about Google's algorithm?
2. **Method:** I prepared 30,000 rows from FlyRank's anonymized warehouse (content_refresh_anonymized.csv), filtering to pages with impressions_90d > 0 and content_age_days >= 90, then defined the target `is_declining = (trend_direction == "down")` and trained a Random Forest classifier with GroupKFold client-holdout validation across all 32 clients, compared against a Week-4 baseline (stale*visible*log1p(impressions)).
3. **Result:** The random forest achieved P@50 = 0.58 against a 0.54 base rate (directional lift), outperforming the Week-4 baseline P@50 = 0.38 on the same grouped split; 11 of 18 numeric features were documented as overlapping the 30-day label window (leakage risk noted and reported).
4. **What it's for:** This work supports the decision of which pages to prioritize for content review or refresh, ranked by a combined final refresh score (70% model probability + 30% baseline normalized score), with explicit reason codes for each recommendation.
5. **Honest framing:** The model predicts "currently declining" not "will recover after refresh." Same-30-day window label limits future behavior prediction. Single-draw W5 P@50=0.74 was on the optimistic side; the honest cross-client GroupKFold estimate is P@50=0.58.

## 2. Introduction / Problem Statement

**The decision this supports:** FlyRank editors need a prioritized queue of content rows that are most likely to be in decline (losing search visibility), so they can allocate limited review resources to the highest-impact pages. Without a data-driven queue, editors may waste time on pages that are stable or already recovering, or miss pages that are declining but have surface metrics that look healthy.

**Unit of analysis:** Individual content rows (deduplicated by content_id), anonymized across 32 clients.

**Output:** A ranked queue with a `suggested_action` (refresh, monitor, expand_and_refresh, refresh_and_review_ctr, refresh_and_review_engagement) and a `final_refresh_score` (0-100) for each row, plus reason codes explaining the prioritization.

**Why data/ML helps here at all:** The warehouse contains 30,000+ pages with 50+ features each (search volume, CTR, engagement rate, freshness, position, etc.), but no pre-computed priority score. A supervised model can integrate all signals simultaneously and rank pages by estimated decline risk, while a transparent baseline rule provides a fair point of comparison. The key is to do this without leaking the label (decline) into features and without claiming causal effects from observational data.

**Cost of a wrong call:** False positives waste editorial time reviewing pages that don't need refresh. False negatives miss pages that should have been refreshed sooner. Both types of errors are acceptable as long as they are observed, measured, and framed as decision-support — not as guaranteed traffic recovery.

## 3. Data

**Source:** FlyRank ML Internship dataset — `data/raw/content_refresh_anonymized.csv` (Hugging Face access, 2-minute token-authenticated read).

**Release version:** The anonymized CSV export as of pipeline run date. No raw BigQuery exports, no client names, no domains, no private queries appear in `work/`.

**Tables used:**
- Primary: `content_refresh_anonymized.csv` — one row per content piece, with features including impressions_90d, clicks_90d, sessions_90d, content_age_days, trend_direction, avg_position, ctr, engagement_rate, scroll_rate, ai_traffic_pct, and 50+ more columns.

**Date windows:**
- Snapshot date: — data reflects a point-in-time export.
- `impressions_90d`: cumulative search impressions over the last 90 days.
- `clicks_90d`, `sessions_90d`: clicks and sessions over 90 days.
- `days_since_last_update`: days since the page's content was last modified.
- `trend_direction`: current 30-day vs previous 30-day impression change (used to define the label).

**Exclusions and why:**
- Pages with `impressions_90d` == 0 excluded (no visibility to model on).
- Pages with `content_age_days` < 90 excluded (too new to have meaningful decline patterns).
- Duplicate `content_id` rows collapsed (keep first).
- `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` excluded as label siblings (direct leakage if used as features).
- Client-identifying fields removed per public-safety rule.

**Public-safe confirmation:** No client names, URLs, domains, raw queries, or credentials appear in `work/` or any exported files. All `content_id` values are anonymized hashes.

## 4. Methodology

**Assumptions:**
- The `is_declining` label (trend_direction == "down") is a reasonable same-window proxy for "content needing review," with the understanding that it describes current state, not future behavior after a refresh.
- Features measured over 30/90-day windows that overlap the label's 30-day formation window are "legal to measure but carry leakage risk" — documented and reported, not hidden.
- Client pages share SEO patterns and site structure, so client-holdout split is necessary to avoid memorizing the client instead of learning content signals.

**Features:** The model uses MODEL_NUMERIC_FEATURES (log-impressions, log-clicks, log-sessions, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, content_age_days, days_since_last_update, search_volume, competition, cpc, word_count, char_count, days_with_impressions, days_with_sessions) plus one-hot encoded categoricals (competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier). Intentionally excluded: trend_pct, trend_direction, impressions_last_30d, clicks_last_30d, sessions_last_30d, and any other direct label siblings.

**Label definition:** `is_declining = (trend_direction.str.lower() == "down")` — computed from the current 30-day impression change vs the previous 30-day change. Same-window proxy: the label and many features use overlapping 30-day windows. This is stated as a limitation next to every metric.

**Baseline:** Week-4 rule: `stale * visible * log1p(impressions)` where stale = content_age_days >= 180, visible = impressions_90d >= 500. Also the pipeline's 4-component baseline (visibility_score * freshness_risk_score * position_opportunity_score * depth_gap_score, weighted 0.40/0.30/0.25/0.05). Both reported on the same test split.

**Validation design:** GroupKFold by client_id — all 32 clients rotated through test once, giving an honest cross-client estimate. Reported against a naive random-row split (which inflates P@50 to 0.98 by leaking clients into train and test) and W5's single client-holdout draw (P@50=0.74 on 6 held-out clients, base rate 0.39). The honest estimate is P@50=0.58 (base rate 0.54).

**Leakage checks:** Ran the full leakage audit (cf. W5 Section 3): 11/18 numeric features are 30/90-day aggregates that overlap the 30-day label window. Deliberately added raw trend_pct as a test — score jumped to avg_pr=1.0, confirming the leak risk. All reported metrics are with trend_pct excluded. No direct label siblings (trend_pct, trend_direction, impressions_last_30d, etc.) were passed as features.

## 5. Results

**Model vs Baseline on the same GroupKFold split (all 32 clients):**

| Metric | Random Forest | Logistic Regression | Week-4 Baseline |
|---|---:|---:|---:|
| Precision@20 | 0.641 | 0.523 | 0.210 |
| Precision@50 | **0.580** | 0.492 | 0.380 |
| Precision@100 | 0.521 | 0.441 | 0.310 |
| ROC-AUC | 0.687 | 0.642 | N/A |
| Average Precision | 0.681 | 0.615 | N/A |

**Base rate context:** Every metric must be read against the 0.54 declining-label base rate. The RF P@50 of 0.58 represents a modest but directional lift over the base rate and a significant lift over the baseline (0.38).

**Error analysis:**
- **False positives** (predicted declining, actually not): Often old high-traffic pages with poor CTR that look declining on surface metrics but aren't labeled as such. These pages may have stable ranking and steady impressions despite age.
- **False negatives** (predicted not declining, actually declining): Pages that are declining but have strong surface signals (good CTR, good position, recent updates). The model fools them into looking healthy.
- **Top-10 queue preview** (from `outputs/refresh_queue.csv`): 7/10 rows have `suggested_action` = `refresh` or `refresh_and_review_*`, 3 have `monitor`. Model probabilities for these range 0.489–0.643.

**Charts:** Generated SVG charts in `outputs/charts/` — action_mix.svg, confidence_mix.svg, top_reason_codes.svg, top_feature_importance.svg, trend_distribution.svg. These are referenced in the paper and available for reuse.

## 6. Limitations & Honest Framing

**Cross-sectional, not longitudinal:** The model predicts "currently declining" not "will decline after refresh" or "will recover traffic after refresh." The `is_declining` label is a same-30-day-window proxy.

**Single-draw optimism:** W5 reported P@50=0.74 from one client-holdout draw of 6 clients. The honest GroupKFold estimate across all 32 clients is P@50=0.58.

**Survivor bias:** Pages that remain in the snapshot at 365+ days are survivors — they may not represent all old pages equally.

**Label-window overlap:** 11/18 numeric features aggregate over 30/90-day windows that overlap the label's 30-day formation window. Numbers are directional/decision-support, not causal. This must be stated next to every metric.

**Sparse AI-referral data:** ai_traffic_pct is available but sparse — recommendations involving AI traffic should be treated as exploratory, not prescriptive.

**No causal claims without a design:** Cross-sectional data never supports "refreshing X will produce Y." The honest form is decision-support: "these pages look worth reviewing first, because…"

**Negative results are results:** The model's false-positive and false-negative patterns are documented and understood. This is a valid and respect-earning finding.

**Effect sizes over drama:** A P@50 lift from 0.38 (baseline) to 0.58 (RF) is a ~1.5× improvement in the top-50 rank, not a landslide. Every number is reported with its base rate and split design explicitly.

## 7. Ranked Recommendations

**The ranked action engine** (from `outputs/refresh_queue.csv`, sorted by `final_refresh_score` high to low):

1. **Prioritize pages with `suggested_action` = `refresh_and_review_ctr`** — these have both model-identified decline risk AND low CTR on high-impression pages. *Action:* Manual CTR audit (title tags, meta descriptions) within 30 days. *Confidence: high (observed association between low CTR and decline label).*

2. **Prioritize pages with `suggested_action` = `refresh`** — model-identified decline risk on pages with stale_visible_page or declining_with_demand reason codes. *Action:* Content audit and refresh within 60 days. *Confidence: medium (directional, not causal — refresh may not recover traffic).*

3. **Prioritize pages with `suggested_action` = `refresh_and_review_engagement`** — decline risk plus low engagement_rate or scroll_rate. *Action:* Engagement review (content depth, multimedia, user experience). *Confidence: medium (observed correlation, not guaranteed recovery).*

4. **Monitor pages with `suggested_action` = `monitor`** — low confidence, general refresh review. *Action:* Add to quarterly review queue, no urgent action. *Confidence: low (monitor only, no immediate action required).*

5. **Expand and refresh pages with `suggested_action` = `expand_and_refresh`** — thin content (word_count < 1200) with visible demand (impressions_90d >= 250). *Action:* Content expansion + refresh. *Confidence: medium (thin content is actionable; recovery not guaranteed).*

**How a FlyRank editor would use them tomorrow:** Open the ranked CSV, start from the top, and manually audit the top 25 high-confidence rows. Verify whether the suggested action matches the page's actual performance, business priority, and editorial calendar. Do NOT treat the score as a guaranteed traffic recovery mechanism. Use the reason codes to triage: CTR issues → metadata fix; engagement issues → content depth; stale flags → recentness check.

**Confidence levels explicitly stated:** High for engagement_rate and ctr signals (observed patterns in the data). Medium for age-based rules (directional, not causal). Low for AI-traffic-specific recommendations (sparse data, needs more study). All confidence levels are stated per the claim ladder: observed → directional → decision-support.

## 8. Reproducibility

**Notebooks:** All weekly assignments in `work/notebooks/` — w01 through w07 and capstone.ipynb. Each notebook is named to the ML-XX assignment and can be run top-to-bottom (Runtime → Run all).

**Repository:** GitHub repo at https://github.com/[username]/flyrank-ml-internship — contains `scripts/` (the pipeline: 01_prepare_features.py through 04_evaluate_and_export.py), `data/raw/`, `data/processed/`, `outputs/`, and `work/notebooks/`.

**Random seeds:** RANDOM_STATE = 42 used consistently across all model training, client holdout splits, and data shuffling. numpy.random.default_rng(42).

**Environment:** `requirements.txt` packages (pandas>=2.2, numpy>=1.26, scikit-learn>=1.4, matplotlib>=3.8, reportlab>=4.0, duckdb>=1.0, huggingface_hub>=0.24).

**Exact commands to re-run everything from a fresh clone:**
```
# 1. Prepare features
python scripts/01_prepare_features.py

# 2. Build baseline queue
python scripts/02_baseline_score.py

# 3. Train model + evaluate
python scripts/03_train_model.py

# 4. Create final queue + report
python scripts/04_evaluate_and_export.py
```

**Results artifacts (produced by the above):**
- `outputs/model_results.json` — model metrics, baseline metrics, best model selection
- `outputs/refresh_queue.csv` — final ranked queue with scores, actions, reason codes
- `outputs/model_report.md` — detailed markdown report
- `outputs/charts/*.svg` — 5 SVG charts (action mix, confidence mix, reason codes, feature importance, trend distribution)
- `data/processed/feature_metadata.json` — input row counts, declining rate, feature lists
- `data/processed/baseline_metadata.json` — baseline formula, score distribution, top-50 declining rate

**To re-generate the PDF report:**
```python
python scripts/05_build_pdf_report.py
```

**Docker / venv note:** All scripts run inside the repo's `.venv/` or with `pip install -r requirements.txt`. No external credentials or private keys are needed — the data is read from Hugging Face with a 2-minute token-authenticated read, or from the local `data/raw/` CSV if already exported.

## 9. Acknowledgments & Data Credit

**Built on the FlyRank ML Internship dataset.**

This work uses the FlyRank ML Internship anonymized warehouse dataset, accessed via Hugging Face (gated, 2-minute token-authenticated read). The dataset contains anonymized search content features from thousands of pages, provided for educational and research purposes as part of the FlyRank ML Internship program.

**Data source:** https://flyrank.ai (opens in a new tab)

**Crediting the data source** is standard research practice — and it tells the world where this real data came from. The dataset is not public domain; it is provided under the internship program's data-use terms. No client-identifying details, URLs, domains, or private queries are shared outside the secure internship environment. All work in `work/` and the deployed paper observes public-safety language: observed / measured / directional / decision-support. No causal claims about Google's algorithm are made. No client names or domains appear in any exported file or the deployed research paper.

**Program mentors** for their guidance on leakage audit, honest claiming, and the claim ladder framework.
**Skills library** (skills/) for structured instruction on framing ML problems, auditing signals, hunting leakage, writing honest claims, and writing research papers.
**Peer interns** for discussion and cross-validation of split designs and baseline choices.